[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aelkharrim/FDM5-3P/blob/main/notebooks/05_scipy.ipynb)

# Python pour la physique — 5. SciPy : le calcul scientifique de haut niveau

> 💡 **Astuce**
>
> **Rappel** : exécutez les cellules dans l'ordre avec **Maj + Entrée**. Les solutions des exercices sont masquées dans des cellules « Solution » : essayez d'abord par vous-même !

In [ ]:
# Cellule de préparation (facultative) : exécutez-la une fois au début.
# Sur Google Colab, NumPy, SciPy, Matplotlib et SymPy sont déjà installés.
import sys
EN_COLAB = 'google.colab' in sys.modules
print('Exécution dans Google Colab' if EN_COLAB else 'Exécution en local')

# Introduction


> 📝 **Note**
>
> Le module `scipy` regroupe des algorithmes de calcul scientifique **de haut niveau**. Il est construit au-dessus de NumPy. Ses principaux sous-modules sont :
>
> | Sous-module          | Contenu                                           |
> | -------------------- | ------------------------------------------------- |
> | `scipy.io`           | entrées/sorties de fichiers                       |
> | `scipy.special`      | fonctions spéciales (Bessel, gamma, erreur…)      |
> | `scipy.linalg`       | algèbre linéaire                                  |
> | `scipy.interpolate`  | interpolation                                     |
> | `scipy.optimize`     | optimisation, recherche de racines, ajustement    |
> | `scipy.stats`        | statistiques et nombres aléatoires                |
> | `scipy.integrate`    | intégration numérique et équations différentielles |
> | `scipy.fft`          | transformées de Fourier rapides                   |
> | `scipy.signal`       | traitement du signal                              |
> | `scipy.ndimage`      | traitement d'images n-dimensionnelles             |


Les sous-modules doivent être importés explicitement, par exemple :

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import scipy.optimize
import scipy.integrate
import scipy.interpolate
import scipy.linalg
import scipy.special
import scipy.stats
import scipy.fft
import scipy.signal

> 💡 **Astuce**
>
> Avant de programmer un algorithme soi-même, **cherchez toujours s'il existe déjà dans SciPy** : il sera presque toujours plus rapide, plus fiable et mieux testé que votre version.

# Entrées/sorties : `scipy.io`

Charger et enregistrer des fichiers MATLAB™ :

In [ ]:
a = np.ones((3, 3))
sp.io.savemat('fichier.mat', {'a': a})  # enregistre un dictionnaire de tableaux
donnees = sp.io.loadmat('fichier.mat')
donnees['a']

Charger des images :

```python
import imageio.v3 as iio
img = iio.imread('fichier.png')   # tableau NumPy
```

Pour les fichiers texte de mesures, `np.loadtxt` (carnet 3) suffit le plus souvent. Pour les tableaux de données mélangeant texte et nombres (CSV, Excel), on utilise la bibliothèque **pandas**.

# Fonctions spéciales : `scipy.special`

Les fonctions spéciales sont des fonctions transcendantes que l'on rencontre souvent en physique : Bessel, gamma, fonction d'erreur, polynômes orthogonaux, intégrales elliptiques, etc.

In [ ]:
import scipy.special as special

# Fonction de Bessel de première espèce, ordre 0 : modes d'une membrane circulaire
x = np.linspace(0, 20, 400)
for n in range(3):
    plt.plot(x, special.jv(n, x), label=f"$J_{n}(x)$")
plt.axhline(0, color='gray', lw=0.5)
plt.legend()
plt.xlabel("x")
plt.show()

La fonction gamma généralise la factorielle ; la fonction d'erreur apparaît dans la loi normale :

In [ ]:
special.gamma(5), special.erf(1.0)

> **Exemple physique : diffraction par une ouverture circulaire**
>
> L'intensité de la figure de diffraction d'une ouverture circulaire (tache d'Airy) est :
>
> $$I(\theta) = I_0 \left[\frac{2 J_1(x)}{x}\right]^2, \qquad x = \frac{\pi D}{\lambda}\sin\theta$$

In [ ]:
x = np.linspace(-15, 15, 1000)
x[x == 0] = 1e-12
I = (2 * special.jv(1, x) / x)**2

plt.semilogy(x, I)
plt.xlabel(r"$x = \pi D \sin\theta / \lambda$")
plt.ylabel(r"$I / I_0$")
plt.title("Tache d'Airy")
plt.ylim(1e-6, 2)
plt.show()

> **À vous de jouer**
>
> Tracez les polynômes de Legendre $P_0, \dots, P_4$ sur $[-1, 1]$ avec `special.eval_legendre(n, x)`. Ils interviennent dans le développement multipolaire du potentiel.

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
x = np.linspace(-1, 1, 200)
for n in range(5):
    plt.plot(x, special.eval_legendre(n, x), label=f"$P_{n}$")
plt.legend()
plt.show()

# Algèbre linéaire : `scipy.linalg`

Le module `scipy.linalg` fournit des fonctionnalités d'algèbre linéaire standard, basées sur les bibliothèques optimisées LAPACK et BLAS (plus complet et plus rapide que `numpy.linalg`).

Le **déterminant** d'une matrice carrée :

In [ ]:
A = np.array([[1, 2],
              [3, 4]])
sp.linalg.det(A)

L'**inverse** :

In [ ]:
iA = sp.linalg.inv(A)
iA

In [ ]:
A @ iA

Pour **résoudre un système linéaire** $A\,\mathbf{x} = \mathbf{b}$, utilisez `solve` (plus précis et plus rapide que de calculer l'inverse) :

In [ ]:
A = np.array([[2., 1.], [1., 3.]])
b = np.array([1., 2.])
x = sp.linalg.solve(A, b)
x, A @ x - b

**Valeurs propres et vecteurs propres** :

In [ ]:
A = np.array([[1, 2],
              [3, 4]])
valeurs, vecteurs = sp.linalg.eig(A)
valeurs

**Décomposition en valeurs singulières (SVD)** :

In [ ]:
U, s, Vh = sp.linalg.svd(A)
s

> **Exemple physique : modes propres de deux oscillateurs couplés**
>
> Deux masses $m$ reliées à deux murs par des ressorts de raideur $k$, et entre elles par un ressort de raideur $k_c$. Les équations du mouvement s'écrivent $m\ddot{\mathbf{x}} = -K\mathbf{x}$, avec
>
> $$K = \begin{pmatrix} k + k_c & -k_c \\ -k_c & k + k_c \end{pmatrix}$$
>
> Les pulsations propres sont $\omega = \sqrt{\lambda/m}$, où $\lambda$ sont les valeurs propres de $K$.

In [ ]:
m, k, kc = 0.5, 20.0, 5.0     # kg, N/m, N/m
K = np.array([[k + kc, -kc],
              [-kc, k + kc]])

lam, modes = sp.linalg.eigh(K)     # eigh : pour les matrices symétriques
omega = np.sqrt(lam / m)
print("Pulsations propres (rad/s) :", omega)
print("Fréquences (Hz)            :", omega / (2 * np.pi))
print("Modes propres (en colonnes) :\n", modes)

Le premier mode `(1, 1)/√2` correspond aux deux masses qui oscillent **en phase** ; le second `(1, -1)/√2` à l'oscillation **en opposition de phase**.

# Interpolation : `scipy.interpolate`

On dispose de mesures en quelques points et on veut estimer la grandeur entre ces points. Créons des données bruitées :

In [ ]:
rng = np.random.default_rng(0)
mesure_t = np.linspace(0, 1, 10)
bruit = (rng.random(10) - 0.5) * 1e-1
mesure_x = np.sin(2 * np.pi * mesure_t) + bruit

`interp1d` construit une fonction d'interpolation :

In [ ]:
from scipy.interpolate import interp1d

lineaire = interp1d(mesure_t, mesure_x)
cubique = interp1d(mesure_t, mesure_x, kind='cubic')

t_fin = np.linspace(0, 1, 100)
plt.plot(mesure_t, mesure_x, 'o', label='mesures')
plt.plot(t_fin, lineaire(t_fin), '-', label='linéaire')
plt.plot(t_fin, cubique(t_fin), '--', label='cubique')
plt.legend()
plt.show()

> ⚠️ **Attention**
>
> L'interpolation **passe exactement par les points**, y compris par le bruit. Pour extraire une loi à partir de mesures bruitées, on préfère un **ajustement** (voir plus loin).


Les splines permettent aussi de dériver et d'intégrer :

In [ ]:
from scipy.interpolate import CubicSpline

x = np.linspace(0, 10, 11)
cs = CubicSpline(x, np.sin(x))
xs = np.linspace(0, 10, 200)

plt.plot(xs, cs(xs), label="spline")
plt.plot(xs, cs(xs, 1), label="dérivée de la spline")
plt.plot(xs, np.cos(xs), ':', label="cos(x) exact")
plt.legend()
plt.show()

# Optimisation et ajustement : `scipy.optimize`

`scipy.optimize` fournit des algorithmes pour la recherche de racines, l'ajustement de courbes et l'optimisation plus générale.

## Recherche de racines

`root_scalar` cherche une racine d'une fonction scalaire (un argument pour lequel la fonction vaut zéro). Comme beaucoup de fonctions de `scipy.optimize`, elle a besoin d'une estimation initiale de la solution, qu'elle affine jusqu'à convergence. Nous fournissons aussi la dérivée pour accélérer la convergence.

In [ ]:
def f(x):
    return (x-1)*(x-2)

def df(x):
    return 2*x - 3

x0 = 0  # estimation initiale
res = sp.optimize.root_scalar(f, x0=x0, fprime=df)
res

> ⚠️ **Attention**
>
> Aucune des fonctions de `scipy.optimize` qui acceptent une estimation initiale n'est garantie de converger pour toutes les estimations possibles ! (Essayez par exemple `x0=1.5` ci-dessus, où la dérivée est exactement nulle.) Dans ce cas, essayez une autre estimation, changez les options (par exemple en donnant un intervalle `bracket`, voir ci-dessous), ou cherchez si SciPy propose une méthode plus adaptée.


Seule la racine en `1.0` est trouvée. En regardant la fonction, on voit qu'il y a une seconde racine en `2.0`. On peut orienter la recherche vers une racine particulière en changeant l'estimation ou en donnant un intervalle qui ne contient que la racine voulue :

In [ ]:
res = sp.optimize.root_scalar(f, bracket=(1.5, 10))
res.root

Pour les problèmes à plusieurs variables, utilisez `scipy.optimize.root` :

In [ ]:
def f(x):
    # intersection du cercle unité et de la droite passant par l'origine
    return [x[0]**2 + x[1]**2 - 1,
            x[1] - x[0]]

res = sp.optimize.root(f, x0=[0, 0])
np.allclose(f(res.x), 0, atol=1e-10)

In [ ]:
np.allclose(res.x, np.sqrt(2)/2)

> **Exemple physique : équation transcendante**
>
> Les niveaux d'énergie d'un puits quantique fini conduisent à des équations comme $\tan(x) = \sqrt{(x_0/x)^2 - 1}$. Un exemple plus simple : à quel instant la position $x(t) = e^{-t}\cos(5t)$ s'annule-t-elle pour la première fois après $t = 0$ ?

In [ ]:
x_t = lambda t: np.exp(-t) * np.cos(5 * t)
res = sp.optimize.root_scalar(x_t, bracket=(0, 0.5))
res.root, np.pi / 10   # on attend 5t = π/2, soit t = π/10

## Ajustement de courbes

Supposons que nous ayons des données sinusoïdales mais bruitées :

In [ ]:
x_data = np.linspace(-5, 5, num=50)  # 50 valeurs entre -5 et 5
bruit = 0.01 * np.cos(100 * x_data)
a, b = 2.9, 1.5
y_data = a * np.cos(b * x_data) + bruit

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(x_data, y_data)
plt.show()

Nous pouvons retrouver l'amplitude, la fréquence et la phase sous-jacentes par un ajustement de courbe aux moindres carrés. Écrivons d'abord une fonction qui prend la variable indépendante en premier argument, et tous les paramètres à ajuster en arguments séparés :

In [ ]:
def f(x, a, b, c):
    return a * np.sin(b * x + c)

Nous utilisons ensuite `scipy.optimize.curve_fit` :

In [ ]:
params, _ = sp.optimize.curve_fit(f, x_data, y_data, p0=[2, 1, 3])
params

In [ ]:
ref = [a, b, np.pi/2]  # ce à quoi on s'attend
np.allclose(params, ref, rtol=1e-3)

Traçons la courbe ajustée sur les données :

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(x_data, y_data, label="Données")
plt.plot(x_data, f(x_data, *params), label="Fonction ajustée")
plt.legend(loc="best")
plt.show()

### Incertitudes sur les paramètres

`curve_fit` renvoie aussi la **matrice de covariance** des paramètres, dont la diagonale donne les variances. On peut ainsi donner un résultat sous la forme « valeur ± incertitude ». Prenons un exemple de **décharge d'un condensateur**, $U(t) = U_0 e^{-t/\tau}$, avec des mesures incertaines :

In [ ]:
rng = np.random.default_rng(3)
t = np.linspace(0, 5, 15)                          # en ms
U_vrai = lambda t, U0, tau: U0 * np.exp(-t / tau)
sigma = 0.15                                       # incertitude sur U (V)
U = U_vrai(t, 5.0, 1.8) + rng.normal(0, sigma, t.size)

popt, pcov = sp.optimize.curve_fit(U_vrai, t, U, p0=[4, 1],
                                   sigma=sigma * np.ones_like(t), absolute_sigma=True)
incert = np.sqrt(np.diag(pcov))
print(f"U0  = {popt[0]:.2f} ± {incert[0]:.2f} V")
print(f"tau = {popt[1]:.2f} ± {incert[1]:.2f} ms")

tt = np.linspace(0, 5, 200)
plt.errorbar(t, U, yerr=sigma, fmt='o', capsize=3, label="mesures")
plt.plot(tt, U_vrai(tt, *popt), label="ajustement")
plt.xlabel("t (ms)"); plt.ylabel("U (V)"); plt.legend()
plt.show()

> **Exercice : températures en Alaska**
>
>
> Les températures extrêmes en Alaska pour chaque mois, à partir de janvier, sont (en degrés Celsius) :
>
> ```text
> max :  17,  19,  21,  28,  33,  38, 37,  37,  31,  23,  19,  18
> min : -62, -59, -56, -46, -32, -18, -9, -13, -25, -46, -52, -58
> ```
>
> 1. Tracez ces extrema de température.
> 2. Définissez une fonction pouvant décrire les températures min et max. Indice : cette fonction doit avoir une période d'un an, et inclure un décalage temporel.
> 3. Ajustez cette fonction aux données avec `curve_fit`.
> 4. Tracez le résultat. L'ajustement est-il raisonnable ? Sinon, pourquoi ?
> 5. Le décalage temporel est-il le même pour les températures min et max, compte tenu de la précision de l'ajustement ?

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
temp_max = np.array([17, 19, 21, 28, 33, 38, 37, 37, 31, 23, 19, 18])
temp_min = np.array([-62, -59, -56, -46, -32, -18, -9, -13, -25, -46, -52, -58])

mois = np.arange(12)
plt.plot(mois, temp_max, "ro")
plt.plot(mois, temp_min, "bo")
plt.xlabel("Mois")
plt.ylabel("Températures min et max")

def temperatures_annuelles(temps, moy, ampl, decalage):
    return moy + ampl * np.cos((temps + decalage) * 2 * np.pi / temps.max())

res_max, cov_max = sp.optimize.curve_fit(temperatures_annuelles, mois, temp_max, [20, 10, 0])
res_min, cov_min = sp.optimize.curve_fit(temperatures_annuelles, mois, temp_min, [-40, 20, 0])

jours = np.linspace(0, 12, num=365)
plt.plot(jours, temperatures_annuelles(jours, *res_max), "r-")
plt.plot(jours, temperatures_annuelles(jours, *res_min), "b-")
plt.show()

print("Décalage (max) :", res_max[2], "±", np.sqrt(cov_max[2, 2]))
print("Décalage (min) :", res_min[2], "±", np.sqrt(cov_min[2, 2]))
# Les deux décalages sont compatibles compte tenu des incertitudes :
# le temps de décalage est le même pour les minima et les maxima.

## Optimisation (recherche de minimum)

Considérons la fonction $f(x) = x^4 - 3x^2 + x$. Ses minima ne sont pas évidents à repérer à vue ; traçons-la :

In [ ]:
def f(x):
    return x**4 - 3 * x**2 + x

x = np.linspace(-2, 2, 200)
plt.plot(x, f(x))
plt.xlabel("x"); plt.ylabel("f(x)")
plt.show()

`scipy.optimize.minimize` trouve un **minimum local** à partir d'une estimation initiale :

In [ ]:
res1 = sp.optimize.minimize(f, x0=-2)
res2 = sp.optimize.minimize(f, x0=2)
print("depuis x0 = -2 :", res1.x, res1.fun)
print("depuis x0 = +2 :", res2.x, res2.fun)

Selon le point de départ, on tombe sur un minimum différent : seul l'un d'eux est le minimum **global**. Pour chercher un minimum global, on peut utiliser par exemple `scipy.optimize.differential_evolution` ou `basinhopping`.


> **Exercice : minimiser une énergie potentielle**
>
> L'énergie d'interaction entre deux atomes est modélisée par le potentiel de Lennard-Jones, $V(r) = 4\varepsilon\left[(\sigma/r)^{12} - (\sigma/r)^6\right]$. Pour $\varepsilon = 1$ et $\sigma = 1$, trouvez numériquement la distance d'équilibre $r_{\min}$ (où $V$ est minimal), et comparez avec la valeur théorique $2^{1/6}\sigma \approx 1{,}122$.

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
def V(r, eps=1.0, sigma=1.0):
    return 4 * eps * ((sigma / r)**12 - (sigma / r)**6)

res = sp.optimize.minimize(lambda r: V(r[0]), x0=[1.5])
print("r_min numérique :", res.x[0])
print("r_min théorique :", 2**(1/6))

# Statistiques et nombres aléatoires : `scipy.stats`

## Distributions statistiques

`scipy.stats` fournit de nombreuses distributions de probabilité. Par exemple, la loi normale :

In [ ]:
from scipy import stats

# Loi normale de moyenne 10 et d'écart-type 2
loi = stats.norm(loc=10, scale=2)
x = np.linspace(2, 18, 200)

plt.plot(x, loi.pdf(x))           # densité de probabilité
plt.xlabel("x"); plt.ylabel("densité")
plt.show()

Chaque distribution offre des méthodes communes : `pdf` (densité), `cdf` (fonction de répartition), `ppf` (inverse de la fonction de répartition), `rvs` (échantillon aléatoire).

In [ ]:
loi.cdf(12), loi.ppf(0.975), loi.rvs(5, random_state=0)

Un exemple fréquent : quelle est la probabilité qu'une mesure s'écarte de la moyenne de plus de deux écarts-types ?

In [ ]:
1 - (stats.norm.cdf(2) - stats.norm.cdf(-2))

## Statistiques d'échantillon et tests d'hypothèses

La distribution des moyennes d'échantillons est plus étroite que celle des données. Comparons deux séries de mesures avec un **test t de Student** :

In [ ]:
rng = np.random.default_rng(42)
serie_a = rng.normal(9.81, 0.10, 30)     # mesures de g, méthode A
serie_b = rng.normal(9.90, 0.10, 30)     # mesures de g, méthode B

print("moyennes :", serie_a.mean(), serie_b.mean())
resultat = stats.ttest_ind(serie_a, serie_b)
resultat

Une *p-value* très petite (typiquement < 0,05) indique que la différence entre les moyennes est **statistiquement significative** : il est peu probable qu'elle soit due au hasard.


> **Exercice**
>
> Générez 20 mesures d'une grandeur de moyenne 5,0 et d'écart-type 0,2. Avec `stats.ttest_1samp`, testez si la moyenne est compatible avec la valeur 5,1. Que concluez-vous ?

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
rng = np.random.default_rng(1)
mesures = rng.normal(5.0, 0.2, 20)
print(stats.ttest_1samp(mesures, 5.1))
# Si la p-value est > 0,05 : on ne peut pas rejeter que la moyenne vaille 5,1.

# Intégration numérique : `scipy.integrate`

## Quadrature

L'intégration d'une fonction se fait avec `quad` :

In [ ]:
from scipy.integrate import quad

# ∫_0^{π/2} sin(x) dx = 1
resultat, erreur = quad(np.sin, 0, np.pi / 2)
resultat, erreur

On peut intégrer jusqu'à l'infini avec `np.inf` :

In [ ]:
# ∫_0^∞ exp(-x²) dx = √π / 2
quad(lambda x: np.exp(-x**2), 0, np.inf), np.sqrt(np.pi) / 2

Quand on dispose de **mesures échantillonnées** plutôt que d'une fonction, on utilise la méthode des trapèzes ou de Simpson :

In [ ]:
t = np.linspace(0, 2, 21)
v = 3 * t**2                       # vitesse mesurée (m/s)
print("trapèzes :", sp.integrate.trapezoid(v, t))
print("Simpson  :", sp.integrate.simpson(v, x=t))
print("exact    :", 2**3)         # ∫ 3t² dt = t³

> **Exemple physique : le moment d'inertie**
>
> Le moment d'inertie d'une tige de longueur $L$ et de masse $M$ autour de son centre est $I = \int_{-L/2}^{L/2} \frac{M}{L}\, x^2\, dx = \frac{ML^2}{12}$. Vérifiez-le numériquement pour $M = 2$ kg, $L = 1{,}5$ m.

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
M, L = 2.0, 1.5
I, _ = quad(lambda x: (M / L) * x**2, -L / 2, L / 2)
print(I, M * L**2 / 12)

## Problèmes à valeur initiale (équations différentielles)

Beaucoup de lois de la physique s'écrivent sous la forme d'équations différentielles ordinaires. `solve_ivp` les résout numériquement. Il faut les écrire comme un **système du premier ordre** $\dot{\mathbf{y}} = F(t, \mathbf{y})$.

**Exemple : l'oscillateur harmonique amorti.** L'équation $\ddot{x} + 2\lambda\dot{x} + \omega_0^2 x = 0$ s'écrit, avec $\mathbf{y} = (x, v)$ :

$$\dot{x} = v, \qquad \dot{v} = -2\lambda v - \omega_0^2 x$$

In [ ]:
from scipy.integrate import solve_ivp

omega0 = 2 * np.pi * 1.0        # pulsation propre (rad/s)
lam = 0.3                       # coefficient d'amortissement (1/s)

def oscillateur(t, y):
    x, v = y
    return [v, -2 * lam * v - omega0**2 * x]

sol = solve_ivp(oscillateur, t_span=(0, 10), y0=[1.0, 0.0],
                t_eval=np.linspace(0, 10, 500))

plt.plot(sol.t, sol.y[0], label="x(t)")
plt.plot(sol.t, sol.y[1], label="v(t)")
plt.xlabel("t (s)")
plt.legend()
plt.show()

Le **portrait de phase** $(x, v)$ montre bien la spirale vers l'équilibre :

In [ ]:
plt.plot(sol.y[0], sol.y[1])
plt.xlabel("x"); plt.ylabel("v")
plt.title("Portrait de phase")
plt.show()

**Le pendule : au-delà des petits angles.** Pour un pendule simple, $\ddot\theta = -\frac{g}{L}\sin\theta$. Comparons la solution exacte à l'approximation harmonique $\sin\theta \approx \theta$ pour un grand angle initial :

In [ ]:
g, L = 9.81, 1.0
theta0 = np.radians(120)

def pendule(t, y):
    theta, omega = y
    return [omega, -(g / L) * np.sin(theta)]

t_eval = np.linspace(0, 6, 600)
sol = solve_ivp(pendule, (0, 6), [theta0, 0.0], t_eval=t_eval, rtol=1e-9)
theta_harm = theta0 * np.cos(np.sqrt(g / L) * t_eval)

plt.plot(t_eval, sol.y[0], label="pendule exact")
plt.plot(t_eval, theta_harm, '--', label="approximation harmonique")
plt.xlabel("t (s)"); plt.ylabel(r"$\theta$ (rad)")
plt.legend()
plt.show()

> **Exercice : chute avec frottements**
>
> Une balle de masse $m = 0{,}1$ kg est lâchée sans vitesse initiale, avec un frottement visqueux $-\alpha v$ ($\alpha = 0{,}2$ kg/s). L'équation est $m\dot v = mg - \alpha v$ (axe vers le bas).
>
> 1. Résolvez-la avec `solve_ivp` entre 0 et 3 s.
> 2. Comparez avec la solution analytique $v(t) = \frac{mg}{\alpha}\left(1 - e^{-\alpha t/m}\right)$.
> 3. Quelle est la vitesse limite ?

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
m, alpha, g = 0.1, 0.2, 9.81
sol = solve_ivp(lambda t, v: g - alpha * v / m, (0, 3), [0.0],
                t_eval=np.linspace(0, 3, 100))
v_exact = (m * g / alpha) * (1 - np.exp(-alpha * sol.t / m))

plt.plot(sol.t, sol.y[0], label="numérique")
plt.plot(sol.t, v_exact, '--', label="analytique")
plt.xlabel("t (s)"); plt.ylabel("v (m/s)"); plt.legend()
plt.show()
print("vitesse limite :", m * g / alpha, "m/s")

# Transformées de Fourier rapides : `scipy.fft`

La **transformée de Fourier** décompose un signal en ses fréquences. Elle est indispensable pour analyser des signaux de mesure (oscilloscope, son, vibrations…).

Créons un signal composé de deux sinusoïdes (50 Hz et 120 Hz) et d'un bruit :

In [ ]:
fs = 1000                                  # fréquence d'échantillonnage (Hz)
t = np.arange(0, 1, 1 / fs)                # 1 seconde de signal
rng = np.random.default_rng(0)
signal = (1.0 * np.sin(2 * np.pi * 50 * t)
          + 0.5 * np.sin(2 * np.pi * 120 * t)
          + 0.4 * rng.standard_normal(t.size))

plt.plot(t[:200], signal[:200])
plt.xlabel("t (s)")
plt.title("Signal temporel (premières 0,2 s)")
plt.show()

La TFD donne les composantes en fréquence. `rfft` (pour signaux réels) et `rfftfreq` (les fréquences correspondantes) :

In [ ]:
spectre = sp.fft.rfft(signal)
freqs = sp.fft.rfftfreq(signal.size, d=1 / fs)
amplitude = 2 * np.abs(spectre) / signal.size

plt.plot(freqs, amplitude)
plt.xlabel("fréquence (Hz)")
plt.ylabel("amplitude")
plt.title("Spectre")
plt.show()

Deux pics nets apparaissent à 50 Hz et 120 Hz, avec des amplitudes proches de 1 et 0,5, même si le signal temporel paraît confus.


> 📝 **Note**
>
> **Théorème de Shannon** : on ne peut détecter que les fréquences inférieures à la moitié de la fréquence d'échantillonnage ($f_s/2$, la fréquence de Nyquist). Au-delà, on observe du *repliement* de spectre (*aliasing*).


On peut aussi **filtrer** dans l'espace de Fourier. Supprimons tout ce qui est au-dessus de 80 Hz :

In [ ]:
spectre_filtre = spectre.copy()
spectre_filtre[freqs > 80] = 0
signal_filtre = sp.fft.irfft(spectre_filtre, n=signal.size)

plt.plot(t[:200], signal[:200], alpha=0.4, label="brut")
plt.plot(t[:200], signal_filtre[:200], label="filtré (< 80 Hz)")
plt.xlabel("t (s)"); plt.legend()
plt.show()

> **Exercice**
>
> Générez un signal de fréquence 200 Hz échantillonné à $f_s = 300$ Hz (donc en dessous de la fréquence de Shannon). À quelle fréquence apparaît le pic dans le spectre ? Expliquez.

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
fs = 300
t = np.arange(0, 2, 1 / fs)
s = np.sin(2 * np.pi * 200 * t)
freqs = sp.fft.rfftfreq(s.size, 1 / fs)
amp = np.abs(sp.fft.rfft(s))
print("pic à", freqs[amp.argmax()], "Hz")   # 100 Hz = |200 - 300| : repliement de spectre

# Traitement du signal : `scipy.signal`

`scipy.signal` propose des outils pour le filtrage, la détection de pics, la convolution, etc.

**Détection de pics** dans un spectre ou une courbe de mesure :

In [ ]:
from scipy.signal import find_peaks

pics, _ = find_peaks(amplitude, height=0.2)
freqs[pics], amplitude[pics]

**Filtrage** : un filtre passe-bas de Butterworth, appliqué à notre signal bruité :

In [ ]:
from scipy.signal import butter, filtfilt

b, a = butter(N=4, Wn=80, btype='low', fs=fs)   # coupure à 80 Hz
signal_butter = filtfilt(b, a, signal)

plt.plot(t[:200], signal[:200], alpha=0.4, label="brut")
plt.plot(t[:200], signal_butter[:200], label="Butterworth 80 Hz")
plt.xlabel("t (s)"); plt.legend()
plt.show()

**Lissage** par moyenne glissante (convolution) :

In [ ]:
fenetre = np.ones(10) / 10
signal_lisse = np.convolve(signal, fenetre, mode='same')

# Traitement d'images : `scipy.ndimage`

Le module `scipy.ndimage` regroupe des opérations sur des tableaux n-dimensionnels, notamment des images : filtres, morphologie mathématique, mesures sur des objets. Un exemple rapide sur une image synthétique :

In [ ]:
from scipy import ndimage

rng = np.random.default_rng(0)
image = np.zeros((64, 64))
image[20:40, 20:40] = 1.0
image += 0.5 * rng.standard_normal(image.shape)
lissee = ndimage.gaussian_filter(image, sigma=2)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(image, cmap='gray');  axes[0].set_title("bruitée")
axes[1].imshow(lissee, cmap='gray'); axes[1].set_title("filtre gaussien")
plt.show()

Pour aller plus loin en traitement d'images, voir la bibliothèque **scikit-image**.

# Exercice de synthèse : analyse d'une expérience

Un oscillateur amorti a été enregistré (position en fonction du temps) avec du bruit. Simulons cette « expérience » :

In [ ]:
rng = np.random.default_rng(7)
t = np.linspace(0, 10, 400)
x_vrai = 2.0 * np.exp(-t / 3.0) * np.cos(2 * np.pi * 0.8 * t + 0.4)
x_mesure = x_vrai + rng.normal(0, 0.08, t.size)
plt.plot(t, x_mesure, '.', ms=3)
plt.xlabel("t (s)"); plt.ylabel("x (cm)")
plt.show()

1. **Ajustez** le modèle $x(t) = A e^{-t/\tau} \cos(2\pi f t + \varphi)$ avec `curve_fit`. Donnez $A$, $\tau$, $f$ et $\varphi$ avec leurs incertitudes.
2. **Vérifiez par la TFD** : le pic du spectre est-il proche de la fréquence ajustée ?
3. **Tracez** les données, l'ajustement et les résidus (différence mesure − modèle) dans deux sous-graphiques.

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
def modele(t, A, tau, f, phi):
    return A * np.exp(-t / tau) * np.cos(2 * np.pi * f * t + phi)

popt, pcov = sp.optimize.curve_fit(modele, t, x_mesure, p0=[1.5, 2, 0.7, 0])
inc = np.sqrt(np.diag(pcov))
for nom, v, u in zip(["A", "tau", "f", "phi"], popt, inc):
    print(f"{nom:4s} = {v:.3f} ± {u:.3f}")

# TFD
freqs = sp.fft.rfftfreq(t.size, d=t[1] - t[0])
spec = np.abs(sp.fft.rfft(x_mesure))
print("pic TFD à", freqs[spec.argmax()], "Hz")

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(7, 6))
ax1.plot(t, x_mesure, '.', ms=3, label="mesures")
ax1.plot(t, modele(t, *popt), 'r', label="ajustement")
ax1.set_ylabel("x (cm)"); ax1.legend()
ax2.plot(t, x_mesure - modele(t, *popt), '.', ms=3)
ax2.axhline(0, color='k', lw=0.5)
ax2.set_ylabel("résidus"); ax2.set_xlabel("t (s)")
plt.show()

# Pour aller plus loin

- La [documentation de SciPy](https://docs.scipy.org/doc/scipy/) et son [tutoriel](https://docs.scipy.org/doc/scipy/tutorial/index.html).
- Le [SciPy Cookbook](https://scipy-cookbook.readthedocs.io) : une collection de recettes.

---

*Fin du carnet 5. Suite : carnet 6 — SymPy (calcul symbolique).*